In [1]:
!pip install -q transformers==4.42.4 datasets accelerate sentencepiece evaluate

import json
import re
import torch

from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(torch.__version__)
print(device)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.4.0a0+f70bd71a48.nv24.06
cuda


In [2]:
def extract_answer(item):

    ans = item.get("answer", {})

    if ans is None:
        return ""

    if isinstance(ans, dict):

        if "answer" in ans and ans["answer"] is not None and len(ans["answer"]) > 0:

            obj = ans["answer"][0]

            if isinstance(obj, dict):

                if "label" in obj:

                    label = obj["label"]

                    if isinstance(label, dict):

                        return str(label.get("en", ""))

                if "name" in obj:

                    return str(obj["name"])

            return str(obj)

        if "mention" in ans:

            return str(ans["mention"])

    return ""


def load_mintaka(path):

    with open(path, "r", encoding="utf-8") as f:

        data = json.load(f)

    questions = []
    answers = []

    for item in data:

        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a = load_mintaka("mintaka_dev.json")
test_q, test_a = load_mintaka("mintaka_test.json")

print(len(train_q))
print(len(dev_q))
print(len(test_q))

14000
2000
4000


In [3]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [4]:
def make_prompt(question):

    prompt = f"""
You are a knowledgeable question answering assistant.

Here are some examples.

Example 1

Question:
Who directed Avatar?

Answer:
James Cameron

Example 2

Question:
Who wrote Harry Potter?

Answer:
J. K. Rowling

Example 3

Question:
Which planet is known as the Red Planet?

Answer:
Mars

Now answer the next question.

Question:
{question}

Answer:
"""

    return prompt


print(make_prompt(train_q[0]))


You are a knowledgeable question answering assistant.

Here are some examples.

Example 1

Question:
Who directed Avatar?

Answer:
James Cameron

Example 2

Question:
Who wrote Harry Potter?

Answer:
J. K. Rowling

Example 3

Question:
Which planet is known as the Red Planet?

Answer:
Mars

Now answer the next question.

Question:
What is the seventh tallest mountain in North America?

Answer:



In [5]:
from tqdm import tqdm

def create_inputs(questions, answers):

    inputs = []
    labels = []

    for q, a in tqdm(zip(questions, answers), total=len(questions)):

        inputs.append(make_prompt(q))
        labels.append(str(a))

    return inputs, labels


train_inputs, train_labels = create_inputs(train_q, train_a)

dev_inputs, dev_labels = create_inputs(dev_q, dev_a)

test_inputs, test_labels = create_inputs(test_q, test_a)

100%|██████████| 4000/4000 [00:00<00:00, 1435913.73it/s]


In [6]:
train_ds = Dataset.from_dict({
    "input_text": train_inputs,
    "target_text": train_labels
})

dev_ds = Dataset.from_dict({
    "input_text": dev_inputs,
    "target_text": dev_labels
})

test_ds = Dataset.from_dict({
    "input_text": test_inputs,
    "target_text": test_labels
})

print(train_ds)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 14000
})


In [7]:
def preprocess(batch):

    model_inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=32,
        truncation=True,
        padding="max_length"
    )

    labels["input_ids"] = [
        [(x if x != tokenizer.pad_token_id else -100) for x in seq]
        for seq in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


train_ds = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names
)

dev_ds = dev_ds.map(
    preprocess,
    batched=True,
    remove_columns=dev_ds.column_names
)

test_ds = test_ds.map(
    preprocess,
    batched=True,
    remove_columns=test_ds.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 5532.92 examples/s]


In [8]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_fewshot",

    learning_rate=3e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    save_total_limit=2,

    fp16=False,

    report_to=[],

    load_best_model_at_end=False
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=dev_ds,

    tokenizer=tokenizer,

    data_collator=data_collator
)

In [10]:
trainer.train()

trainer.save_model("./flan_t5_fewshot")

tokenizer.save_pretrained("./flan_t5_fewshot")

Epoch,Training Loss,Validation Loss
1,1.992000,1.640305
2,1.810600,1.599874
3,1.870300,1.594186


('./flan_t5_fewshot/tokenizer_config.json',
 './flan_t5_fewshot/special_tokens_map.json',
 './flan_t5_fewshot/spiece.model',
 './flan_t5_fewshot/added_tokens.json',
 './flan_t5_fewshot/tokenizer.json')

In [11]:
def predict_answer(question):

    prompt = make_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction.strip()


print(predict_answer(test_q[0]))

William Henry Harrison


In [12]:
predictions = []

for question in tqdm(test_q):

    predictions.append(
        predict_answer(question)
    )

print(predictions[:10])

100%|██████████| 4000/4000 [04:04<00:00, 16.34it/s]

['William Henry Harrison', '1', 'Drake', '2', 'True', 'Bill Belichick', 'False', '3', '1980', 'The Adventures of Tom Sawyer']


In [13]:
from sklearn.metrics import accuracy_score, f1_score

In [14]:
def normalize(text):

    text = str(text).lower()

    text = re.sub(r"[^\w\s]", "", text)

    text = " ".join(text.split())

    return text

In [15]:
gold = [normalize(x) for x in test_a]

pred = [normalize(x) for x in predictions]

correct = 0

for g, p in zip(gold, pred):

    if g == p:

        correct += 1

hit1 = correct / len(gold)

mrr = hit1

hit5 = hit1

accuracy = accuracy_score(gold, pred)

f1 = f1_score(
    gold,
    pred,
    average="weighted"
)

print(f"Hit@1     : {hit1:.4f}")
print(f"Hit@5     : {hit5:.4f}")
print(f"MRR        : {mrr:.4f}")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

Hit@1     : 0.2225
Hit@5     : 0.2225
MRR        : 0.2225
Accuracy : 0.2225
F1 Score : 0.1780


In [16]:
import pandas as pd

results = pd.DataFrame({

    "Question": test_q,

    "Ground Truth": test_a,

    "Prediction": predictions
})

results.to_csv(
    "fewshot_predictions.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
